# Schema Contracts for CDC Pipeline

This notebook defines expected schema contracts for all tables in the CDC pipeline.
These contracts are used for schema drift detection and validation.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, 
    LongType, DecimalType, TimestampType, BooleanType
)

In [ ]:
# ============================================================================
# POSTGRESQL SOURCE SCHEMAS (Debezium CDC Events)
# ============================================================================

# Expected schema for Debezium CDC events from 'orders' table
# This represents the payload.after / payload.before structure
ORDER_CDC_PAYLOAD_SCHEMA = StructType([
    StructField("payload", StructType([
        StructField("before", StructType([
            StructField("id", IntegerType(), True),
            StructField("product_id", IntegerType(), True),
            StructField("product_name", StringType(), True),  # Denormalized
            StructField("price", StringType(), True),
            StructField("created_at", LongType(), True)
        ]), True),
        StructField("after", StructType([
            StructField("id", IntegerType(), True),
            StructField("product_id", IntegerType(), True),
            StructField("product_name", StringType(), True),  # Denormalized
            StructField("price", StringType(), True),
            StructField("created_at", LongType(), True)
        ]), True),
        StructField("op", StringType(), False),  # c=create, u=update, d=delete, r=read
        StructField("ts_ms", LongType(), True),
        StructField("transaction", StructType([
            StructField("id", StringType(), True),
            StructField("total_order", LongType(), True),
            StructField("data_collection_order", LongType(), True)
        ]), True)
    ]), True)
])

# Expected schema for Debezium CDC events from 'products' table
PRODUCT_CDC_PAYLOAD_SCHEMA = StructType([
    StructField("payload", StructType([
        StructField("before", StructType([
            StructField("id", IntegerType(), True),
            StructField("product_name", StringType(), True),
            StructField("weight", StringType(), True),
            StructField("color", StringType(), True),
            StructField("created_at", LongType(), True),
            StructField("updated_at", LongType(), True)
        ]), True),
        StructField("after", StructType([
            StructField("id", IntegerType(), True),
            StructField("product_name", StringType(), True),
            StructField("weight", StringType(), True),
            StructField("color", StringType(), True),
            StructField("created_at", LongType(), True),
            StructField("updated_at", LongType(), True)
        ]), True),
        StructField("op", StringType(), False),
        StructField("ts_ms", LongType(), True),
        StructField("transaction", StructType([
            StructField("id", StringType(), True),
            StructField("total_order", LongType(), True),
            StructField("data_collection_order", LongType(), True)
        ]), True)
    ]), True)
])

In [ ]:
# ============================================================================
# BRONZE LAYER SCHEMAS (Raw CDC Events)
# ============================================================================

# Bronze orders - raw Kafka events with metadata
BRONZE_ORDERS_SCHEMA = StructType([
    StructField("topic", StringType(), False),
    StructField("partition", IntegerType(), False),
    StructField("offset", LongType(), False),
    StructField("kafka_timestamp", TimestampType(), True),
    StructField("message_key", StringType(), True),
    StructField("value", StringType(), True),  # JSON string
    StructField("source_schema", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("ingested_at", TimestampType(), False)
])

# Bronze products - raw Kafka events with metadata
BRONZE_PRODUCTS_SCHEMA = StructType([
    StructField("topic", StringType(), False),
    StructField("partition", IntegerType(), False),
    StructField("offset", LongType(), False),
    StructField("kafka_timestamp", TimestampType(), True),
    StructField("message_key", StringType(), True),
    StructField("value", StringType(), True),  # JSON string
    StructField("source_schema", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("ingested_at", TimestampType(), False)
])

In [ ]:
# ============================================================================
# SILVER LAYER SCHEMAS (Cleansed, Conformed)
# ============================================================================

# Silver orders - current state with audit columns
SILVER_ORDERS_SCHEMA = StructType([
    StructField("id", IntegerType(), False),
    StructField("product_id", IntegerType(), True),
    StructField("product_legacy", StringType(), True),
    StructField("price", DecimalType(12, 2), False),
    StructField("created_at", TimestampType(), True),
    StructField("last_inserted_dt", TimestampType(), True),
    StructField("last_updated_dt", TimestampType(), True)
])

# Silver products - current state with audit columns
SILVER_PRODUCTS_SCHEMA = StructType([
    StructField("id", IntegerType(), False),
    StructField("product_name", StringType(), False),
    StructField("weight", DecimalType(10, 2), False),
    StructField("color", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True),
    StructField("last_inserted_dt", TimestampType(), True),
    StructField("last_updated_dt", TimestampType(), True)
])

In [ ]:
# ============================================================================
# GOLD LAYER SCHEMAS (Business-Ready)
# ============================================================================

# Gold aggregate by product and color
GOLD_TOTAL_PRODUCT_ORDERS_SCHEMA = StructType([
    StructField("product_name", StringType(), False),
    StructField("color", StringType(), False),
    StructField("total_amount", DecimalType(22, 2), False)
])

In [ ]:
# ============================================================================
# SCHEMA CONTRACT REGISTRY
# ============================================================================

# Central registry of all schema contracts
SCHEMA_CONTRACTS = {
    # Source CDC schemas
    "cdc.orders": ORDER_CDC_PAYLOAD_SCHEMA,
    "cdc.products": PRODUCT_CDC_PAYLOAD_SCHEMA,
    
    # Bronze layer schemas
    "bronze.orders": BRONZE_ORDERS_SCHEMA,
    "bronze.products": BRONZE_PRODUCTS_SCHEMA,
    
    # Silver layer schemas
    "silver.orders": SILVER_ORDERS_SCHEMA,
    "silver.products": SILVER_PRODUCTS_SCHEMA,
    
    # Gold layer schemas
    "gold.total_products_order": GOLD_TOTAL_PRODUCT_ORDERS_SCHEMA,
}

# Schema validation policies per layer
SCHEMA_POLICIES = {
    "bronze": {
        "policy": "additive_only",  # Allow new columns, block removals
        "alert_channel": "all",
        "block_on_critical": True
    },
    "silver": {
        "policy": "strict",  # Block on any breaking change
        "alert_channel": "all",
        "block_on_critical": True
    },
    "gold": {
        "policy": "strict",  # Never allow schema drift in gold
        "alert_channel": "all",
        "block_on_critical": True
    }
}


def get_schema_contract(contract_name: str) -> StructType:
    """
    Get a schema contract by name.
    
    Args:
        contract_name: Name of the contract (e.g., 'silver.orders')
    
    Returns:
        StructType schema contract
    
    Raises:
        KeyError: If contract name not found
    """
    if contract_name not in SCHEMA_CONTRACTS:
        available = ", ".join(SCHEMA_CONTRACTS.keys())
        raise KeyError(
            f"Schema contract '{contract_name}' not found. "
            f"Available contracts: {available}"
        )
    return SCHEMA_CONTRACTS[contract_name]


def get_layer_policy(layer: str) -> dict:
    """
    Get schema validation policy for a layer.
    
    Args:
        layer: One of 'bronze', 'silver', 'gold'
    
    Returns:
        Policy dictionary
    """
    if layer not in SCHEMA_POLICIES:
        raise KeyError(f"Policy for layer '{layer}' not found")
    return SCHEMA_POLICIES[layer]

## Usage Example

```python
# Run this notebook to load schema contracts
%run ./helpers/NB_schema_contracts

# Get a specific contract
orders_contract = get_schema_contract('silver.orders')

# Get policy for a layer
silver_policy = get_layer_policy('silver')

# Use with schema drift validation
%run ./helpers/NB_schema_drift_helpers

is_valid, drift = validate_schema_with_policy(
    spark,
    expected_schema=orders_contract,
    actual_schema=actual_schema,
    table_name="silver.orders",
    policy=silver_policy['policy'],
    alert_channel=silver_policy['alert_channel']
)
```